# Validate Metadata

Validates selected standardized Work metadata before promotion to Silver. The notebook verifies canonical identifiers and references, checks referential integrity against the current valid parent state, reconciles validated records with their Bronze source snapshots, and publishes trusted metadata to Silver.

## Input Parameters

In [1]:
p_source_database = "Sales_Operational"
p_execution_step_id = 0
p_source_execution_step_ids = "{}"
p_metadata_entities = "[]"
p_created_by = ""

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 3, Finished, Available, Finished, False)

## Imports

In [2]:
import json
from datetime import datetime, timezone

from pyspark.sql import functions as F

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 4, Finished, Available, Finished, False)

## Helper Functions

Defines reusable functions for loading metadata, applying validation rules, resolving validation dependencies, reconciling results, preparing Silver metadata, and determining process status.

In [4]:
def get_validation_condition(df):
    """
    Returns a Boolean Spark condition that is True only when all
    temporary validation flags in the DataFrame have passed.
    """

    # Identify all validation result flags created during the notebook.
    validation_flag_columns = [
        column_name
        for column_name in df.columns
        if column_name.startswith("is_")
        and column_name.endswith("_passed")
    ]

    # Start as True so each validation flag can be combined with AND.
    condition = F.lit(True)

    for column_name in validation_flag_columns:
        condition = condition & F.col(column_name)

    return condition

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 6, Finished, Available, Finished, False)

In [ ]:
def load_work_metadata(
    source_database,
    metadata_entities
):
    """
    Loads the current Work metadata candidates for the selected
    source database and metadata entities.
    """

    work_dfs = {}

    for entity_name in metadata_entities:

        work_dfs[entity_name] = (
            spark.table(f"work.{entity_name}")
            .filter(
                F.col("source_database") == source_database
            )
        )

    return work_dfs

In [ ]:
def load_reference_metadata(
    source_database,
    entity_name
):
    """
    Loads a Silver metadata entity used only as a validation dependency.
    """

    return (
        spark.table(f"silver.{entity_name}")
        .filter(
            F.col("source_database") == source_database
        )
    )

In [ ]:
def get_parent_reference(
    work_dfs,
    source_database,
    parent_entity,
    parent_column
):
    """
    Returns the valid parent identifiers required for referential
    integrity validation.

    Uses the current Work entity when it is part of the current process;
    otherwise uses the trusted Silver state.
    """

    if parent_entity in work_dfs:
        parent_df = (
            work_dfs[parent_entity]
            .filter(
                get_validation_condition(
                    work_dfs[parent_entity]
                )
            )
        )
    else:
        parent_df = load_reference_metadata(
            source_database=source_database,
            entity_name=parent_entity
        )

    return (
        parent_df
        .select(parent_column)
        .distinct()
    )

In [ ]:
def determine_validation_outcome(work_dfs):
    """
    Combines all validation flags and returns only rows that passed
    every validation rule for Silver promotion.
    """

    valid_dfs = {}

    for entity_name, df in work_dfs.items():

        df = df.withColumn(
            "is_validation_passed",
            get_validation_condition(df)
        )

        valid_dfs[entity_name] = (
            df
            .filter(F.col("is_validation_passed"))
        )

        work_dfs[entity_name] = df

    return work_dfs, valid_dfs

In [ ]:
def add_validation_result(
    validation_results,
    details,
    affected_row_count,
    validation_code
):
    validation_results.append({
        "details": details,
        "affected_row_count": affected_row_count,
        "validation_code": validation_code
    })

## Validate Required IDs

Verifies that each metadata entity has its required canonical identifier. Rows with missing identifiers are flagged and excluded from Silver promotion.

In [8]:
required_id_validations = {
    "metadata_objects": "object_id",
    "metadata_columns": "column_id",
    "metadata_constraints": "constraint_id",
    "metadata_constraint_columns": "constraint_column_id",
    "metadata_relationships": "relationship_id",
    "metadata_modules": "module_id"
}

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 10, Finished, Available, Finished, False)

In [9]:
def validate_required_ids(
    work_dfs,
    validation_results,
    validations
):
    for entity_name, df in work_dfs.items():

        id_column = validations[entity_name]

        df = df.withColumn(
            "is_required_id_passed",
            F.col(id_column).isNotNull()
        )

        affected_row_count = (
            df
            .filter(~F.col("is_required_id_passed"))
            .count()
        )

        add_validation_result(
            validation_results=validation_results,
            details=f"Required ID validation: {entity_name}.{id_column} contains null values.",
            affected_row_count=affected_row_count,
            validation_code="NOT_NULL"
        )

        work_dfs[entity_name] = df

    return work_dfs

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 11, Finished, Available, Finished, False)

## Validate ID Uniqueness

Verifies that canonical identifiers uniquely identify metadata records. All rows participating in duplicated identifiers are flagged as invalid.

In [12]:
uniqueness_validations = {
    "metadata_objects": "object_id",
    "metadata_columns": "column_id",
    "metadata_constraints": "constraint_id",
    "metadata_constraint_columns": "constraint_column_id",
    "metadata_relationships": "relationship_id",
    "metadata_modules": "module_id"
}

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 14, Finished, Available, Finished, False)

In [13]:
def validate_id_uniqueness(
    work_dfs,
    validation_results,
    validations
):
    """
    Validates that canonical identifiers are unique within each metadata entity.
    """

    for entity_name, df in work_dfs.items():

        id_column = validations[entity_name]

        duplicate_ids = (
            df
            .groupBy(id_column)
            .count()
            .filter(F.col("count") > 1)
            .select(id_column)
        )

        flag_column = f"is_{id_column}_unique_passed"

        df = (
            df.alias("d")
            .join(
                duplicate_ids.alias("x"),
                F.col(f"d.{id_column}") == F.col(f"x.{id_column}"),
                "left"
            )
            .withColumn(
                flag_column,
                F.col(f"x.{id_column}").isNull()
            )
            .select(
                "d.*",
                flag_column
            )
        )

        affected_row_count = (
            df
            .filter(~F.col(flag_column))
            .count()
        )

        add_validation_result(
            validation_results=validation_results,
            details=f"Uniqueness validation: duplicate {id_column} values found in {entity_name}.",
            affected_row_count=affected_row_count,
            validation_code="DUPLICATE"
        )

        work_dfs[entity_name] = df

    return work_dfs

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 15, Finished, Available, Finished, False)

## Validate Required References

Verifies that required canonical reference identifiers are populated after metadata standardization. This validation checks presence only; referential integrity is validated separately.

In [10]:
required_reference_validations = {
    "metadata_columns": [
        "object_id"
    ],
    "metadata_constraints": [
        "object_id"
    ],
    "metadata_constraint_columns": [
        "constraint_id",
        "column_id"
    ],
    "metadata_modules": [
        "object_id"
    ]
}

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 12, Finished, Available, Finished, False)

In [11]:
def validate_required_references(
    work_dfs,
    validation_results,
    validations
):
    """
    Validates that required canonical reference identifiers are not null.
    """

    for entity_name, df in work_dfs.items():

        reference_columns = validations.get(entity_name, [])

        for id_column in reference_columns:

            flag_column = f"is_{id_column}_not_null_passed"

            df = df.withColumn(
                flag_column,
                F.col(id_column).isNotNull()
            )

            affected_row_count = (
                df
                .filter(~F.col(flag_column))
                .count()
            )

            add_validation_result(
                validation_results=validation_results,
                details=f"Required reference validation: {entity_name}.{id_column} contains null values.",
                affected_row_count=affected_row_count,
                validation_code="NOT_NULL"
            )

        work_dfs[entity_name] = df

    return work_dfs

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 13, Finished, Available, Finished, False)

## Validate Referential Integrity

Verifies that canonical reference identifiers resolve to valid parent metadata records. Unresolved references are flagged to prevent orphan records from being promoted to Silver.

In [14]:
referential_integrity_validations = {
    "metadata_columns": [
        ("object_id", "metadata_objects", "object_id")
    ],
    "metadata_constraints": [
        ("object_id", "metadata_objects", "object_id")
    ],
    "metadata_constraint_columns": [
        ("constraint_id", "metadata_constraints", "constraint_id"),
        ("column_id", "metadata_columns", "column_id")
    ],
    "metadata_modules": [
        ("object_id", "metadata_objects", "object_id")
    ]
}

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 16, Finished, Available, Finished, False)

In [15]:
def validate_referential_integrity(
    work_dfs,
    validation_results,
    validations,
    source_database
):
    """
    Validates that required child references exist in the current
    valid parent state.
    """

    for child_entity, child_df in work_dfs.items():

        reference_rules = validations.get(child_entity, [])

        for (
            child_column,
            parent_entity,
            parent_column
        ) in reference_rules:

            parent_df = get_parent_reference(
                work_dfs=work_dfs,
                source_database=source_database,
                parent_entity=parent_entity,
                parent_column=parent_column
            )

            flag_column = f"is_{child_column}_fk_check_passed"

            child_df = (
                child_df.alias("c")
                .join(
                    F.broadcast(parent_df).alias("p"),
                    F.col(f"c.{child_column}") == F.col(f"p.{parent_column}"),
                    "left"
                )
                .withColumn(
                    flag_column,
                    F.col(f"p.{parent_column}").isNotNull()
                )
                .select(
                    "c.*",
                    flag_column
                )
            )

            affected_row_count = (
                child_df
                .filter(~F.col(flag_column))
                .count()
            )

            add_validation_result(
                validation_results=validation_results,
                details=(
                    f"Referential integrity validation: "
                    f"{child_entity}.{child_column} has values not found in "
                    f"{parent_entity}.{parent_column}."
                ),
                affected_row_count=affected_row_count,
                validation_code="FK_CHECK"
            )

        work_dfs[child_entity] = child_df

    return work_dfs

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 17, Finished, Available, Finished, False)

## Reconcile Row Counts

Compares the number of Bronze source records processed with the number of validated records eligible for Silver. The target count intentionally excludes rows that failed validation.

In [17]:
def reconcile_row_counts(
    valid_dfs,
    source_database,
    source_execution_step_ids
):
    """
    Reconciles Bronze source row counts against validated rows
    prepared for Silver promotion.
    """

    reconciliation_results = []

    for entity_name, target_df in valid_dfs.items():

        source_row_count = (
            spark.table(f"bronze.{entity_name}")
            .filter(
                (F.col("source_database") == source_database) &
                (
                    F.col("execution_step_id") == source_execution_step_ids[entity_name]
                )
            )
            .count()
        )

        target_row_count = target_df.count()

        reconciliation_results.extend([
            {
                "metric_name": "ROW_COUNT",
                "reconciliation_key": entity_name.upper(),
                "reconciliation_side": "SOURCE",
                "metric_value_bigint": source_row_count
            },
            {
                "metric_name": "ROW_COUNT",
                "reconciliation_key": entity_name.upper(),
                "reconciliation_side": "TARGET",
                "metric_value_bigint": target_row_count
            }
        ])

    return reconciliation_results

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 19, Finished, Available, Finished, False)

## Prepare Silver Metadata

Removes temporary validation flags from valid records and adds Silver execution lineage and audit metadata before promotion.

In [20]:
def prepare_silver_metadata(
    valid_dfs,
    execution_step_id,
    created_at,
    created_by
):
    """
    Removes temporary validation flags and adds Silver lineage
    and audit metadata.
    """

    silver_ready_dfs = {}

    for entity_name, df in valid_dfs.items():

        validation_flag_columns = [
            column_name
            for column_name in df.columns
            if column_name.startswith("is_")
            and column_name.endswith("_passed")
        ]

        silver_ready_dfs[entity_name] = (
            df
            .drop(
                *validation_flag_columns,
                "is_validation_passed"
            )
            .withColumn(
                "execution_step_id",
                F.lit(execution_step_id).cast("long")
            )
            .withColumn(
                "created_at",
                F.lit(created_at)
            )
            .withColumn(
                "created_by",
                F.lit(created_by)
            )
        )

    return silver_ready_dfs

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 22, Finished, Available, Finished, False)

## Load Valid Metadata into Silver

Publishes validated metadata to Silver using scoped snapshot replacement for the selected source database.

In [21]:
def load_valid_metadata_to_silver(
    silver_ready_dfs,
    source_database
):
    """
    Publishes validated metadata to Silver using scoped snapshot replacement.
    """

    for entity_name, source_df in silver_ready_dfs.items():

        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .option(
                "replaceWhere",
                f"source_database = '{source_database}'"
            )
            .saveAsTable(f"silver.{entity_name}")
        )

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 23, Finished, Available, Finished, False)

## Determine Process Status

Evaluates validation and reconciliation results to determine whether the process completed cleanly or requires observation.

In [22]:
def determine_process_status(
    validation_results,
    reconciliation_results
):
    """
    Determines whether the process completed with validation or
    reconciliation observations.
    """

    has_validation_issues = any(
        result["affected_row_count"] > 0
        for result in validation_results
    )

    reconciliation_by_key = {}

    for result in reconciliation_results:
        key = result["reconciliation_key"]
        side = result["reconciliation_side"]

        reconciliation_by_key.setdefault(key, {})[side] = (
            result["metric_value_bigint"]
        )

    has_reconciliation_issues = any(
        values.get("SOURCE") != values.get("TARGET")
        for values in reconciliation_by_key.values()
    )

    return (
        has_validation_issues
        or has_reconciliation_issues
    )

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 24, Finished, Available, Finished, False)

## Execute Validation Process

Orchestrates the validation workflow for the selected metadata entities, from Work loading and validation through reconciliation, Silver preparation, promotion, and final process status evaluation.

In [ ]:
created_at = datetime.now(timezone.utc)

p_source_execution_step_ids = json.loads(
    p_source_execution_step_ids
)

p_metadata_entities = json.loads(
    p_metadata_entities
)

validation_results = []

In [ ]:
work_dfs = load_work_metadata(
    source_database=p_source_database,
    metadata_entities=p_metadata_entities
)

work_dfs = validate_required_ids(
    work_dfs=work_dfs,
    validation_results=validation_results,
    validations=required_id_validations
)

work_dfs = validate_required_references(
    work_dfs=work_dfs,
    validation_results=validation_results,
    validations=required_reference_validations
)

work_dfs = validate_id_uniqueness(
    work_dfs=work_dfs,
    validation_results=validation_results,
    validations=uniqueness_validations
)

work_dfs = validate_referential_integrity(
    work_dfs=work_dfs,
    validation_results=validation_results,
    validations=referential_integrity_validations,
    source_database=p_source_database
)

work_dfs, valid_dfs = determine_validation_outcome(
    work_dfs=work_dfs
)

reconciliation_results = reconcile_row_counts(
    valid_dfs=valid_dfs,
    source_database=p_source_database,
    source_execution_step_ids=p_source_execution_step_ids
)

silver_ready_dfs = prepare_silver_metadata(
    valid_dfs=valid_dfs,
    execution_step_id=p_execution_step_id,
    created_timestamp=created_at,
    created_by=p_created_by
)

load_valid_metadata_to_silver(
    silver_ready_dfs=silver_ready_dfs,
    source_database=p_source_database
)

has_observations = determine_process_status(
    validation_results=validation_results,
    reconciliation_results=reconciliation_results
)

## Output

In [24]:
result = {
    "validation_results": validation_results,
    "reconciliation_results": reconciliation_results,
    "has_observations": has_observations
}

mssparkutils.notebook.exit(
    json.dumps(result)
)

StatementMeta(, 5e9a4fb8-1aaa-4699-8821-4c0359c47be3, 26, Finished, Available, Finished, False)

ExitValue: {"validation_results": [{"details": "Required ID validation: metadata_objects.object_id contains null values.", "affected_row_count": 0, "validation_code": "NOT_NULL"}, {"details": "Required ID validation: metadata_columns.column_id contains null values.", "affected_row_count": 0, "validation_code": "NOT_NULL"}, {"details": "Required ID validation: metadata_constraints.constraint_id contains null values.", "affected_row_count": 0, "validation_code": "NOT_NULL"}, {"details": "Required ID validation: metadata_constraint_columns.constraint_column_id contains null values.", "affected_row_count": 0, "validation_code": "NOT_NULL"}, {"details": "Required ID validation: metadata_relationships.relationship_id contains null values.", "affected_row_count": 0, "validation_code": "NOT_NULL"}, {"details": "Required ID validation: metadata_modules.module_id contains null values.", "affected_row_count": 0, "validation_code": "NOT_NULL"}, {"details": "Required reference validation: metadata_